In [1]:
import pandas as pd

In [4]:
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/flybase/"

# mapping

In [10]:
import pandas as pd

file = pd.read_csv(
    f'{BASE_PATH}flybase/fbal_to_fbgn_fb_2024_02.tsv',
    sep='\t',
    comment='#',
    header=None,
    names=['AlleleID', 'AlleleSymbol', 'GeneID', 'GeneSymbol'],
    low_memory=False
)

# print(file.head())
file= file.drop_duplicates()
file
Droso_allele_2_Gene_Sym_dict = file.set_index('AlleleID')['GeneID'].to_dict()
# Droso_allele_2_Gene_Sym_dict

# Flybase

In [20]:

Gene_pheno = pd.read_csv(
    f'{BASE_PATH}flybase/genotype_phenotype_data_fb_2024_02.tsv',
    sep='\t',
    comment='#',
    header=None,
    names=[
        'genotype_symbols',
        'genotype_FBids',
        'phenotype_name',
        'phenotype_id',
        'qualifier_names',
        'qualifier_ids',
        'reference'
    ],
    low_memory=False
)

Gene_pheno = Gene_pheno.drop(Gene_pheno.index[-1])
Gene_pheno['genotype_FBids'] = Gene_pheno['genotype_FBids'].str.split('/').str[0]
Gene_pheno['genotype_FBids'] = Gene_pheno['genotype_FBids'].str.split().str[0]
Gene_pheno['genotype_symbols'] = Gene_pheno['genotype_symbols'].str.split('[').str[0]
Gene_pheno['Gene_ID'] = Gene_pheno['genotype_FBids'].map(Droso_allele_2_Gene_Sym_dict)
Gene_pheno


,genotype_symbols,genotype_FBids,phenotype_name,phenotype_id,qualifier_names,qualifier_ids,reference,Gene_ID
0,064Ya,FBal0119724,chemical sensitive,FBcv:0000440,NaN,NaN,FBrf0131396,FBgn0043467
1,1.1.3,FBal0190078,abnormal eye color,FBcv:0000355,NaN,NaN,FBrf0190779,FBgn0084807
2,1.1.3,FBal0190078,pigment cell,FBbt:00004230,NaN,NaN,FBrf0190779,FBgn0084807
3,106y,FBal0151008,fertile,FBcv:0000374,NaN,NaN,FBrf0141372,FBgn0067637
4,106y,FBal0151008,viable,FBcv:0000349,NaN,NaN,FBrf0141372,FBgn0067637
...,...,...,...,...,...,...,...,...
387620,zyd,FBal0282649,abnormal behavior,FBcv:0000387,adult stage|recessive,FBdv:00005369|FBcv:0000298,FBrf0220564,FBgn0265767
387621,zyd,FBal0263592,abnormal behavior,FBcv:0000387,third instar larval stage,FBdv:00005339,FBrf0220564,FBgn0265767
387622,zyd,FBal0219878,abnormal behavior,FBcv:0000387,third instar larval stage,FBdv:00005339,FBrf0220564,FBgn0265767
387623,zyd,FBal0219878,abnormal behavior,FBcv:0000387,NaN,NaN,FBrf0220564,FBgn0265767


In [21]:
Gene_pheno = Gene_pheno[['genotype_symbols','phenotype_name']]
Gene_pheno = Gene_pheno.drop_duplicates()
#Gene_pheno = Gene_pheno.groupby('genotype_symbols')['phenotype_name'].agg(','.join).reset_index()
# Rename columns
Gene_pheno.rename(columns={'genotype_symbols': 'Head', 'phenotype_name': 'Tail'}, inplace=True)

# Add new columns
Gene_pheno['Head_type'] = 'Gene'
Gene_pheno['Tail_type'] = 'Phenotype'
Gene_pheno['Relation'] = 'Gene_Phenotype'
Gene_pheno['KG_Source'] = 'FlyBase'

Gene_pheno['Head_ID_IS'] = 'FlyBase_symbol'
Gene_pheno['Tail_ID_IS'] = ''


# Reorder columns
Gene_pheno = Gene_pheno[['Head', 'Relation', 'Tail', 'Head_type', 'Tail_type','KG_Source', 'Head_ID_IS', 'Tail_ID_IS']]
Gene_pheno

,Head,Relation,Tail,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,064Ya,Gene_Phenotype,chemical sensitive,Gene,Phenotype,FlyBase,FlyBase_symbol,
1,1.1.3,Gene_Phenotype,abnormal eye color,Gene,Phenotype,FlyBase,FlyBase_symbol,
2,1.1.3,Gene_Phenotype,pigment cell,Gene,Phenotype,FlyBase,FlyBase_symbol,
3,106y,Gene_Phenotype,fertile,Gene,Phenotype,FlyBase,FlyBase_symbol,
4,106y,Gene_Phenotype,viable,Gene,Phenotype,FlyBase,FlyBase_symbol,
...,...,...,...,...,...,...,...,...
387609,zwi,Gene_Phenotype,some die during embryonic stage,Gene,Phenotype,FlyBase,FlyBase_symbol,
387610,zyd,Gene_Phenotype,abnormal neurophysiology,Gene,Phenotype,FlyBase,FlyBase_symbol,
387611,zyd,Gene_Phenotype,viable,Gene,Phenotype,FlyBase,FlyBase_symbol,
387612,zyd,Gene_Phenotype,bang sensitive,Gene,Phenotype,FlyBase,FlyBase_symbol,


In [23]:
Gene_pheno.to_csv(f'{OUT_PATH}Flybase_Droso_Gene_Phenotype.csv',index = None)